<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20261013/%E3%82%AF%E3%83%A9%E3%82%B9%E3%82%BF%E3%83%AA%E3%83%B3%E3%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# クラスタリング ― K-means法による顧客セグメンテーション

本資料では、クラスタリングの基本的な考え方、データの前処理、K-means法の実行、クラスタ数の検討、結果の解釈までを一連の流れとして学ぶ。


#クラスタリングの概要

##クラスタリングとは

クラスタリングとは、特徴が似ている観測対象を複数のグループに分ける手法である。クラスタリングは**教師なし学習**の代表的な方法である。

本資料では、各顧客を「年齢」と「利用回数」という2つの特徴量で表した
サンプルデータを用いる。


##ビジネスでの利用例

クラスタリングは、例えば次の用途で利用される。

- 顧客セグメンテーション
- 店舗や商品の類型化
- 購買パターンの把握
- 異常な取引や行動の候補抽出
- 文書・画像・行動ログのグループ化

ただし、計算で得られたクラスタをそのまま利用するのではなく、
業務知識、施策可能性、再現性などの観点も加味して
クラスタの妥当性を確認する必要がある。


#実行環境の準備

本資料はGoogle Colabでの実行を想定する。
以下のセルでは、必要なライブラリを読み込み、日本語を含むグラフを表示できるようにする。


In [ ]:
%%capture
# 日本語の表示環境のインストール
!pip install japanize-matplotlib

In [ ]:
# 必要なライブラリの準備
import sys
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#前処理として標準化を行う。そのためのモジュールをインポートする
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# 日本語表示用ライブラリを利用
import japanize_matplotlib

# 表示設定
pd.options.display.float_format = "{:.2f}".format
sns.set_theme(style='whitegrid', font='IPAexGothic')

plt.rcParams["figure.figsize"] = (8, 5)
#マイナス表示に Unicode U+2212 を使わない（ASCII 半角ハイフン U+002D を使用させる）
plt.rcParams["axes.unicode_minus"] = False

# 再現性を確保するための乱数シード
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

#データの読み込みと品質確認

##利用するデータ

「利用回数.xlsx」には、少なくとも次の3列が含まれている。

| 列名 | 内容 | クラスタリングへの利用 |
|---|---|---|
| 顧客ID | 顧客を識別するための番号 | 利用しない |
| 年齢 | 顧客の年齢 | 利用する |
| 利用回数 | 一定期間内の利用回数 | 利用する |

顧客IDは識別子であり、値の近さに実質的な意味がないため、特徴量として用いてはならない。


##ファイルの読み込み


In [ ]:
# インターネット上にあるファイルを指定する
url = 'https://raw.githubusercontent.com/yajima-yasutoshi/shunan-u/refs/heads/main/data/%E5%88%A9%E7%94%A8%E5%9B%9E%E6%95%B0.csv'
# 今回は、df という変数に読み込むことにする
df = pd.read_csv(url)

##データ構造の確認

行数・列数、列名、データ型、欠損値の有無を確認する。


In [ ]:
print(f"行数: {df.shape[0]:,}")
print(f"列数: {df.shape[1]}")
#display(df.head())
df.info()

In [ ]:
# 記述統計量の確認
df.describe(include="all").T

##必須列とデータ品質の確認

分析前に、必須列、欠損値、重複、数値型への変換可能性を確認する。

In [ ]:
required_columns = ["顧客ID", "年齢", "利用回数"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"必須列が不足している: {missing_columns}")

# 年齢と利用回数を数値型に変換する
for col in ["年齢", "利用回数"]:
    df[col] = pd.to_numeric(df[col])

quality_summary = pd.DataFrame({
    "欠損数": df[required_columns].isna().sum(),
    "欠損率": df[required_columns].isna().mean(),
    "ユニーク数": df[required_columns].nunique(dropna=True)
})
display(quality_summary)

print(f"完全重複行数: {df.duplicated().sum():,}")
print(f"顧客IDの重複数: {df['顧客ID'].duplicated().sum():,}")

df_clean = (
    df.drop_duplicates()
      .dropna(subset=["年齢", "利用回数"])
      .copy()
)

本演習では、説明を簡潔にするため、データには問題が無いとして進める。
実務では、欠損や重複があれば、その生じた原因を確認したうえで、補完・修正・除外の方針を決定する必要がある。


#探索的データ分析

##分布の確認

クラスタリングの前に、各特徴量の分布、外れ値、値の範囲を確認する。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_clean["年齢"].dropna(), bins=10)
axes[0].set_title("年齢の分布")
axes[0].set_xlabel("年齢")
axes[0].set_ylabel("件数")

axes[1].hist(df_clean["利用回数"].dropna(), bins=10)
axes[1].set_title("利用回数の分布")
axes[1].set_xlabel("利用回数")
axes[1].set_ylabel("件数")

plt.tight_layout()
plt.show()


##散布図による関係の確認

2つの数値変数の関係を確認するには散布図が有効である。点の密集や分離の程度を観察する。以下の図では、２るいは３個の集まりに分類できそうなデータであることが分かる。ただし、実施にはデータは多くの変数を持つため、常に視覚的にクラスタが観測できるわけではない

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_clean, x="年齢", y="利用回数", s=60)
plt.title("年齢と利用回数の散布図")
plt.show()


##外れ値の確認

K-means法は平均値をクラスタ中心として用いるため、外れ値の影響を受けやすい。箱ひげ図などを用いて、極端な値がないかを確認する。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df_clean, y="年齢", ax=axes[0])
axes[0].set_title("年齢の箱ひげ図")

sns.boxplot(data=df_clean, y="利用回数", ax=axes[1])
axes[1].set_title("利用回数の箱ひげ図")

plt.tight_layout()
plt.show()


#標準化

##標準化が必要となる理由

K-means法は、観測対象間の距離を用いてクラスタを形成する。特徴量ごとに単位や値の範囲が大きく異なる場合、数値の大きい特徴量が距離計算を支配する。

一般に、**距離に基づく手法で特徴量の尺度が異なる場合は、原則として標準化を行う**。ただし、同一尺度であり、元の単位に分析上の意味がある場合などには、標準化しない選択もあり得る。


##標準化の定義

標準化では、各値 $x$ を次式で変換する。

$$
z = \frac{x - \mu}{\sigma}
$$

ここで、$\mu$ は平均、$\sigma$ は標準偏差である。
これにより、変換後のデータは、平均が0、標準偏差が1となる。


In [ ]:
features = ["年齢", "利用回数"]
X_original = df_clean[features].copy()

display(
    X_original.agg(["min", "max", "mean", "std", "var"]).T
)


## 前処理関数を用いた標準化

Sklearnには、データ分析で必要となる前処理を行うための関数が準備されている。

以下を参照せよ。
https://scikit-learn.org/stable/modules/preprocessing.html#


In [ ]:
#インスタンスの作成と初期化
scaler = StandardScaler()

#変換に必要となる平均・分散を計算する
scaler.fit(X_original)

#実際に変換処理を行い結果を得る
#結果はDataFrameでなく、数値arrayであることに注意
X_scaled_array = scaler.transform(X_original)

#結果をDataFrameに変換する
X_scaled = pd.DataFrame(
    X_scaled_array,
    columns=X_original.columns,
    index=X_original.index
)

X_scaled

In [ ]:
#インスタンスの作成と初期化
scaler = StandardScaler()

# fit_transformは、平均・分散の計算と変換を連続して実行する
# 結果はDataFrameでなく、数値arrayであることに注意
X_scaled_array = scaler.fit_transform(X_original)

X_scaled = pd.DataFrame(
    X_scaled_array,
    columns=features,
    index=X_original.index
)

`StandardScaler`が保持する平均と分散は、`mean_`と`var_`で確認できる。


In [ ]:
display(scaler.mean_)
display(scaler.var_)

In [ ]:
#DataFrame に格納して見やすく表示する
scaler_parameters = pd.DataFrame({
    "平均": scaler.mean_,
    "分散": scaler.var_,
    "標準偏差": np.sqrt(scaler.var_)
}, index=features)

display(scaler_parameters)


In [ ]:
#変換後のデータで平均と標準偏差などを確認
standardized_summary = pd.DataFrame({
    "平均": X_scaled.mean(),
    "標準偏差": X_scaled.std(),
    "最小値": X_scaled.min(),
    "最大値": X_scaled.max()
})
display(standardized_summary)

##標準化前後の比較

点の分布の範囲が変化していることを確認する。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(
    data=df_clean, x="年齢", y="利用回数",
    ax=axes[0], s=60
)
axes[0].set_title("標準化前")

sns.scatterplot(
    data=X_scaled, x="年齢", y="利用回数",
    ax=axes[1], s=60
)
axes[1].set_title("標準化後")

plt.tight_layout()
plt.show()


#K-means法の実行

##K-means法の考え方

K-means法は、あらかじめ指定したクラスタ数 $K$ に基づいて、次の処理を反復する。

1. $K$ 個のクラスタ中心を初期化する。
2. 各データを、最も近いクラスタ中心に割り当てる。
3. 各クラスタに属するデータの平均を、新しいクラスタ中心とする。
4. 割り当てまたはクラスタ中心がほぼ変化しなくなるまで繰り返す。

K-means法は、各データと所属クラスタ中心との**二乗距離の総和**を小さくすることを目的とする。


##主なパラメータ

| パラメータ | 内容 |
|---|---|
| `n_clusters` | 作成するクラスタ数 |
| `init` | 初期中心の決め方。通常は`k-means++`を用いる |
| `n_init` | 異なる初期値で計算する回数 |
| `max_iter` | 1回の計算における最大反復回数 |
| `random_state` | 乱数の再現性を確保する値 |

初期値によって結果が変わる可能性があるため、`random_state`を固定して再現性を確保する。


##クラスタ数を3として実行

ここでは説明のため、クラスタ数を3と仮定する。適切なクラスタ数の検討方法は後に扱う。


In [ ]:
n_clusters = 3

#インスタンスの初期化
kmeans = KMeans(
    n_clusters=n_clusters,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=RANDOM_SEED
)

#クラスタリングの実行とラベル付け
cluster_labels = kmeans.fit_predict(X_scaled)

#クラスタ番号を追加したDataFrameを作成する
df_clustered = df_clean.copy()
df_clustered["cluster"] = cluster_labels

display(df_clustered.head())


#クラスタ結果の可視化と解釈

##散布図による可視化


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_clustered,
    x="年齢",
    y="利用回数",
    hue="cluster",
    palette="Set1",
    s=70
)
plt.title("K-means法によるクラスタリング結果")
plt.legend(title="クラスタ")
plt.show()


##クラスタ中心

K-means法のクラスタ中心は、標準化後の空間で計算される。解釈を容易にするため、`inverse_transform`を用いて元の単位に戻す。


In [ ]:
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)

cluster_centers = pd.DataFrame(
    centers_original,
    columns=features
)
cluster_centers["cluster"] = range(n_clusters)

display(cluster_centers[["cluster"] + features])


In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df_clustered,
    x="年齢",
    y="利用回数",
    hue="cluster",
    palette="Set1",
    s=60,
    alpha=0.75
)

plt.scatter(
    cluster_centers["年齢"],
    cluster_centers["利用回数"],
    marker="X",
    s=250,
    edgecolor="black",
    linewidth=1.5,
    label="クラスタ中心"
)

plt.title("クラスタリング結果とクラスタ中心")
plt.legend(title="クラスタ")
plt.show()


##クラスタ別の件数と平均

クラスタの特徴を解釈する際には、平均値だけでなく、件数、割合、ばらつきも確認する必要がある。


In [ ]:
cluster_profile = (
    df_clustered
    .groupby("cluster")
    .agg(
        顧客数=("顧客ID", "count"),
        年齢平均=("年齢", "mean"),
        年齢標準偏差=("年齢", "std"),
        利用回数平均=("利用回数", "mean"),
        利用回数標準偏差=("利用回数", "std")
    )
)

cluster_profile["構成比"] = (
    cluster_profile["顧客数"] / cluster_profile["顧客数"].sum()
)

display(cluster_profile)


##解釈の手順

クラスタの解釈は、次の順序で行う。

1. クラスタごとの件数と構成比を確認する。
2. 各特徴量の平均・中央値・ばらつきを確認し、他のクラスタのそれらと比較する
3. 「高年齢・高頻度」など、データに即した仮の名称を付ける。
4. 元データの具体的な顧客を確認し、解釈に矛盾がないかを検証する。
5. 施策対象として意味があるかを業務部門と確認する。

クラスタ番号そのものには意味がない。
データや乱数設定が変われば番号が入れ替わる場合がある。


#クラスタ数の検討

クラスタ数は、単一の指標だけで自動的に決めるものではない。統計的な指標と業務上の解釈可能性を組み合わせて検討する。


##クラスタ内平方和（Inertia）

クラスタ内平方和とは、各データと所属クラスタ中心との二乗距離の総和である。値が小さいほどクラスタ内のまとまりは強いが、クラスタ数を増やせば原則として小さくなる。

クラスタ内平方和は`inertia_`に格納されている。


In [ ]:
print(f"クラスタ数3のInertia: {kmeans.inertia_:.3f}")

##エルボー法

クラスタ数を決める最も代表的な方法の一つに**エルボー法**がある。
クラスタ数を増やしたときのInertiaの減少を図示し、減少幅が大きく変わる「肘」に相当する点を候補とする。エルボーが明確に現れない場合もあり、判断には主観が入る。


In [ ]:
max_k = min(10, len(X_scaled) - 1)

inertia_values = []
k_candidates = range(1, max_k + 1)

for k in k_candidates:
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_SEED
    )
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

elbow_df = pd.DataFrame({
    "クラスタ数": list(k_candidates),
    "Inertia": inertia_values
})

plt.figure(figsize=(8, 5))
sns.lineplot(
    data=elbow_df,
    x="クラスタ数",
    y="Inertia",
    marker="o"
)
plt.title("エルボー法")
plt.xticks(list(k_candidates))
plt.show()


##シルエット係数

シルエット係数は、同じクラスタ内での近さと、他クラスタからの離れ具合を組み合わせた指標である。値は概ね-1から1の範囲を取り、値が大きいほどクラスタが分離していることを示す。

ただし、シルエット係数が最大のクラスタ数が、必ずしも業務上最適であるとは限らない。


### シルエット係数の計算方法

各データ点 $i$ について、シルエット係数 $s(i)$ は以下の手順で計算される。

1.  **データ点 $i$ と同じクラスタ内の他の点との平均距離 $a(i)$ の計算（凝集度）：**
    $a(i)$ は、データ点 $i$ と同じクラスタに属する**他のすべての**データ点とのユークリッド距離の平均です。この値が小さいほど、データ点 $i$ は自身のクラスタに密接に結合している（凝集度が高い）ことを意味する。

2.  **データ点 $i$ と最も近い隣接クラスタの点との平均距離 $b(i)$ の計算（分離度）：**
    $b(i)$ は、データ点 $i$ が属していない
    **最も近い（最も平均距離が小さい）**
    クラスタに属するすべてのデータ点とのユークリッド距離の平均です。この値が大きいほど、データ点 $i$ は隣接するクラスタから十分に離れている（分離度が高い）ことを意味する。

3.  **シルエット係数 $s(i)$ の計算：**
    データ点 $i$ のシルエット係数 $s(i)$ は以下の式で計算される。

    $$ s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} $$

    *   $s(i)$ の値が1に近い場合：$b(i)$ が $a(i)$ よりも非常に大きく、データ点 $i$ は自身のクラスタに密接にまとまっており、他のクラスタから遠く離れていることを示す。
    *   $s(i)$ の値が0に近い場合：$a(i)$ と $b(i)$ がほぼ同じであり、データ点 $i$ がクラスタの境界に近いことを示す。
    *   $s(i)$ の値が-1に近い場合：$a(i)$ が $b(i)$ よりも大きく、データ点 $i$ が間違ったクラスタに割り当てられている可能性が高いことを示す。

**クラスタ全体のシルエット係数**は、すべてのデータ点のシルエット係数 $s(i)$ の平均値として計算される。
この平均値が高いほど、クラスタリング全体が良好であると評価される。

In [ ]:
silhouette_records = []

for k in range(2, max_k + 1):
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_SEED
    )
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_records.append({
        "クラスタ数": k,
        "シルエット係数": score
    })

silhouette_df = pd.DataFrame(silhouette_records)
display(silhouette_df)

plt.figure(figsize=(8, 5))
sns.lineplot(
    data=silhouette_df,
    x="クラスタ数",
    y="シルエット係数",
    marker="o"
)
plt.title("クラスタ数とシルエット係数")
plt.xticks(silhouette_df["クラスタ数"])
plt.show()


##最終的なクラスタ数の決め方

クラスタ数は、次の観点を総合して決定する。

- エルボー法で減少幅が変化する位置
- シルエット係数
- 各クラスタの件数が極端に少なくないか
- クラスタの特徴を説明できるか
- 施策をクラスタ別に実施できるか
- 時期や標本が変わっても概ね再現するか

本データでクラスタ数3を採用する場合も、「グラフが3に見えるから」だけではなく、上記の観点を明示する必要がある。


##採用するクラスタ数で再学習

以下では、最終的に採用するクラスタ数を`BEST_K`に指定して再学習する。


In [ ]:
BEST_K = 3

final_model = KMeans(
    n_clusters=BEST_K,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=RANDOM_SEED
)

df_final = df_clean.copy()
df_final["cluster"] = final_model.fit_predict(X_scaled)

final_centers = pd.DataFrame(
    scaler.inverse_transform(final_model.cluster_centers_),
    columns=features
)
final_centers["cluster"] = range(BEST_K)

display(
    df_final.groupby("cluster")[features]
            .agg(["count", "mean", "std", "median"])
)


#K-means法を利用する際の注意点

K-means法には次の性質と制約がある。

1. **クラスタ数を事前に指定する必要がある。**
2. **初期値によって局所的な解に到達する可能性がある。** → そのため初期値をランダムに変更し、クラスタ内距離が小さくなるものを採用する。
3. **外れ値の影響を受けやすい。**
5. **尺度が異なる特徴量では標準化が重要である。**
6. **カテゴリ変数をそのまま距離計算に用いることはできない。**
7. **高次元では距離の差が分かりにくくなる場合がある。**
8. **計算上のまとまりと、業務上意味のある区分は一致しない場合がある。**

データの形状や目的によっては、階層型クラスタリング、DBSCAN、Gaussian Mixture Model、k-prototypesなどの別手法も検討する。


##実務で確認すべき事項

- 特徴量の選択理由は説明できるか。
- 欠損値・外れ値・重複の処理は妥当か。
- クラスタ数の決定根拠は複数あるか。
- クラスタが時期を変えても安定しているか。
- 特定の属性に不利益を与える運用にならないか。
- クラスタ別施策の効果を検証できる設計になっているか。
- クラスタ名が偏見や過度な単純化を招かないか。

クラスタリングは探索の出発点であり、施策の妥当性を保証するものではない。


#Wineデータを用いた演習

##演習の目的

scikit-learnに含まれるWineデータを用いて、複数の特徴量を持つデータに対するクラスタリングを実行する。Wineデータには13個の化学的特徴量と、参考用の品種ラベルが含まれている。

クラスタリングでは品種ラベルを学習に使用しない。品種ラベルは、クラスタリング結果を事後的に比較する場合に限って用いる。


In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()

wine_df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

print(f"行数: {wine_df.shape[0]}")
print(f"特徴量数: {wine_df.shape[1]}")
display(wine_df.head())


##特徴量の概要


In [ ]:
wine_description = pd.DataFrame({
    "特徴量": wine.feature_names,
    "日本語の概要": [
        "アルコール",
        "リンゴ酸",
        "灰分",
        "灰分のアルカリ度",
        "マグネシウム",
        "総フェノール",
        "フラバノイド",
        "非フラバノイドフェノール",
        "プロアントシアニン",
        "色の強度",
        "色相",
        "希釈ワインのOD280/OD315",
        "プロリン"
    ]
})

display(wine_description)


##分析の手順

次の手順で分析せよ。

1. 欠損値と基本統計量を確認する。
2. 全13特徴量を標準化する。
3. クラスタ数2～6について、Inertiaとシルエット係数を計算する。
4. 指標をもとにクラスタ数を決定する。
5. 採用したクラスタ数でK-means法を実行する。
6. クラスタごとの特徴量平均を比較し、特徴を説明する。
7. 正解ラベルを学習には用いず、最後に参考としてクラスタとの対応を確認する。


##分析例


In [ ]:
# 1. データ品質と基本統計量の確認
print(f"欠損数: {wine_df.isna().sum().sum()}")
display(wine_df.describe().T)


In [ ]:
# 2. 標準化
wine_scaler = StandardScaler()
wine_scaled = wine_scaler.fit_transform(wine_df)


In [ ]:
# 3. クラスタ数ごとの評価
wine_records = []

for k in range(2, 7):
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=RANDOM_SEED
    )
    labels = model.fit_predict(wine_scaled)
    wine_records.append({
        "クラスタ数": k,
        "Inertia": model.inertia_,
        "シルエット係数": silhouette_score(wine_scaled, labels)
    })

wine_evaluation = pd.DataFrame(wine_records)
display(wine_evaluation)


In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=wine_evaluation,
    x="クラスタ数",
    y="Inertia",
    marker="o"
)
plt.title("Wineデータ：エルボー法")
plt.xticks(wine_evaluation["クラスタ数"])
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=wine_evaluation,
    x="クラスタ数",
    y="シルエット係数",
    marker="o"
)
plt.title("Wineデータ：シルエット係数")
plt.xticks(wine_evaluation["クラスタ数"])
plt.show()


このデータでは、クラスタ数3が有力な候補となる。ただし、指標の値だけでなく、クラスタ別の特徴と解釈可能性を確認する必要がある。


In [ ]:
# 4. クラスタ数3で学習
WINE_K = 3

wine_model = KMeans(
    n_clusters=WINE_K,
    n_init=10,
    random_state=RANDOM_SEED
)

wine_labels = wine_model.fit_predict(wine_scaled)

wine_result = wine_df.copy()
wine_result["cluster"] = wine_labels

display(wine_result.head())


In [ ]:
# 5. クラスタごとの平均値
wine_profile = wine_result.groupby("cluster").mean().T
display(wine_profile)


特徴量ごとに単位と値の範囲が異なるため、クラスタの比較には標準化後の平均も有効である。正の値は全体平均より高く、負の値は全体平均より低いことを表す。


In [ ]:
wine_scaled_df = pd.DataFrame(
    wine_scaled,
    columns=wine.feature_names
)
wine_scaled_df["cluster"] = wine_labels

wine_profile_scaled = wine_scaled_df.groupby("cluster").mean().T
display(wine_profile_scaled)


##品種ラベルとの事後比較

Wineデータには既知の品種ラベルが含まれている。
上で行ったクラスタリングの学習には使用していない。
以下は、クラスタと品種ラベルの対応を事後的に確認するための集計である。


In [ ]:
comparison = pd.crosstab(
    pd.Series(wine.target, name="既知の品種ラベル"),
    pd.Series(wine_labels, name="クラスタ番号")
)

display(comparison)


#まとめ

本講義では、クラスタリング分析に必要な一連の手順を扱った。

1. 分析目的と特徴量を明確にする。
2. データ型、欠損値、重複、外れ値を確認する。
3. 距離計算への影響を考慮して標準化する。
4. K-means法を実行する。
5. クラスタ中心とクラスタ別統計量から特徴を解釈する。
6. エルボー法とシルエット係数を用いてクラスタ数を検討する。
7. 統計的妥当性だけでなく、業務上の意味を確認する。


## 参考資料

- scikit-learn: Clustering: https://scikit-learn.org/stable/modules/clustering.html

ライブラリの仕様は更新されるため、実行環境のバージョンに応じて公式ドキュメントを確認すること。
